# Word Embeddings: Word2Vec (CBOW vs. Skip-Gram) and AvgWord2Vec

Train Word2Vec on a real NLTK corpus (Gutenberg), compare CBOW vs. Skip-Gram nearest neighbours, then build Average Word2Vec sentence features and use them in a simple classifier on a subset of `fetch_20newsgroups`.

In [1]:
import nltk
nltk.download("gutenberg", quiet=True)
nltk.download("punkt", quiet=True)
nltk.download("punkt_tab", quiet=True)
nltk.download("stopwords", quiet=True)
print("NLTK resources ready.")

NLTK resources ready.


## Training corpus

Use `nltk.corpus.gutenberg`'s *Alice in Wonderland* text, tokenized into sentences of lowercase word tokens, as a modest built-in corpus for Word2Vec.

In [2]:
from nltk.corpus import gutenberg
from nltk.tokenize import word_tokenize

raw_text = gutenberg.raw("carroll-alice.txt")
sentences_raw = nltk.sent_tokenize(raw_text)

tokenized_corpus = [
    [w.lower() for w in word_tokenize(s) if w.isalpha()]
    for s in sentences_raw
]
tokenized_corpus = [s for s in tokenized_corpus if len(s) >= 3]

print(f"Number of sentences: {len(tokenized_corpus)}")
print("Example sentence tokens:", tokenized_corpus[10])

Number of sentences: 1491
Example sentence tokens: ['she', 'took', 'down', 'a', 'jar', 'from', 'one', 'of', 'the', 'shelves', 'as', 'she', 'passed', 'it', 'was', 'labelled', 'orange', 'marmalade', 'but', 'to', 'her', 'great', 'disappointment', 'it', 'was', 'empty', 'she', 'did', 'not', 'like', 'to', 'drop', 'the', 'jar', 'for', 'fear', 'of', 'killing', 'somebody', 'so', 'managed', 'to', 'put', 'it', 'into', 'one', 'of', 'the', 'cupboards', 'as', 'she', 'fell', 'past', 'it']


## Train Word2Vec: CBOW (`sg=0`) and Skip-Gram (`sg=1`)

In [3]:
from gensim.models import Word2Vec

w2v_cbow = Word2Vec(
    sentences=tokenized_corpus, vector_size=100, window=5, min_count=3, sg=0, epochs=20, seed=42
)
w2v_skipgram = Word2Vec(
    sentences=tokenized_corpus, vector_size=100, window=5, min_count=3, sg=1, epochs=20, seed=42
)

print(f"CBOW vocabulary size: {len(w2v_cbow.wv.key_to_index)}")
print(f"Skip-Gram vocabulary size: {len(w2v_skipgram.wv.key_to_index)}")

CBOW vocabulary size: 1014
Skip-Gram vocabulary size: 1014


## Compare `most_similar` results between CBOW and Skip-Gram

In [4]:
query_words = ["alice", "queen"]

for word in query_words:
    if word in w2v_cbow.wv.key_to_index:
        print(f"--- Most similar to '{word}' (CBOW) ---")
        for sim_word, score in w2v_cbow.wv.most_similar(word, topn=5):
            print(f"  {sim_word:15s} {score:.3f}")
    if word in w2v_skipgram.wv.key_to_index:
        print(f"--- Most similar to '{word}' (Skip-Gram) ---")
        for sim_word, score in w2v_skipgram.wv.most_similar(word, topn=5):
            print(f"  {sim_word:15s} {score:.3f}")
    print()

--- Most similar to 'alice' (CBOW) ---
  so              0.973
  doubt           0.970
  grown           0.970
  that            0.969
  this            0.969
--- Most similar to 'alice' (Skip-Gram) ---
  but             0.748
  feeling         0.744
  rather          0.743
  child           0.731
  cautiously      0.730

--- Most similar to 'queen' (CBOW) ---
  with            0.987
  his             0.984
  voice           0.981
  rabbit          0.977
  white           0.974
--- Most similar to 'queen' (Skip-Gram) ---
  executioner     0.847
  king            0.824
  knave           0.816
  hearts          0.807
  pointing        0.795



Skip-Gram typically produces more semantically coherent neighbours on a small corpus like this because it generates far more (center word, context word) training pairs per sentence, giving rarer words more gradient updates, whereas CBOW's context-averaging trains faster but needs more data to specialise.

## Average Word2Vec sentence vectors

Represent a document as the mean of its in-vocabulary word vectors, using the CBOW model.

In [5]:
import numpy as np

def avg_word2vec(tokens, model, vector_size=100):
    vectors = [model.wv[w] for w in tokens if w in model.wv.key_to_index]
    if not vectors:
        return np.zeros(vector_size)
    return np.mean(vectors, axis=0)

example_vec = avg_word2vec(tokenized_corpus[10], w2v_cbow)
print(f"Sentence: {tokenized_corpus[10]}")
print(f"AvgWord2Vec vector shape: {example_vec.shape}")
print(example_vec[:10])

Sentence: ['she', 'took', 'down', 'a', 'jar', 'from', 'one', 'of', 'the', 'shelves', 'as', 'she', 'passed', 'it', 'was', 'labelled', 'orange', 'marmalade', 'but', 'to', 'her', 'great', 'disappointment', 'it', 'was', 'empty', 'she', 'did', 'not', 'like', 'to', 'drop', 'the', 'jar', 'for', 'fear', 'of', 'killing', 'somebody', 'so', 'managed', 'to', 'put', 'it', 'into', 'one', 'of', 'the', 'cupboards', 'as', 'she', 'fell', 'past', 'it']
AvgWord2Vec vector shape: (100,)
[ 0.31854674  0.16402343 -0.08255834 -0.03749483 -0.1119201   0.10080513
  0.26621112  0.2973343  -0.24414852  0.07136585]


## AvgWord2Vec as features for a classifier

Train a fresh Word2Vec model on a small labelled subset of `fetch_20newsgroups` (2 categories), build AvgWord2Vec features for each document, and train a `LogisticRegression` classifier on top.

In [6]:
from sklearn.datasets import fetch_20newsgroups

categories = ["rec.sport.hockey", "sci.space"]
news = fetch_20newsgroups(subset="train", categories=categories, remove=("headers", "footers", "quotes"))
news_test = fetch_20newsgroups(subset="test", categories=categories, remove=("headers", "footers", "quotes"))

print(f"Train docs: {len(news.data)}, Test docs: {len(news_test.data)}")
print(f"Categories: {news.target_names}")

Train docs: 1193, Test docs: 793
Categories: ['rec.sport.hockey', 'sci.space']


In [7]:
from nltk.corpus import stopwords

stop_words = set(stopwords.words("english"))

def tokenize_doc(doc):
    return [w.lower() for w in word_tokenize(doc) if w.isalpha() and w.lower() not in stop_words]

train_tokens = [tokenize_doc(d) for d in news.data]
test_tokens = [tokenize_doc(d) for d in news_test.data]

print("Example tokenized doc:", train_tokens[0][:15])

Example tokenized doc: ['individual', 'leaders', 'total', 'points', 'final', 'standings', 'note', 'games', 'played', 'points', 'per', 'games', 'accurate', 'player', 'team']


In [8]:
# Train Word2Vec on the training documents themselves (a modest, task-specific corpus)
news_w2v = Word2Vec(sentences=train_tokens, vector_size=100, window=5, min_count=2, sg=1, epochs=20, seed=42)
print(f"News Word2Vec vocabulary size: {len(news_w2v.wv.key_to_index)}")

News Word2Vec vocabulary size: 9141


In [9]:
X_train = np.array([avg_word2vec(toks, news_w2v) for toks in train_tokens])
X_test = np.array([avg_word2vec(toks, news_w2v) for toks in test_tokens])
y_train, y_test = news.target, news_test.target

print(f"X_train shape: {X_train.shape}, X_test shape: {X_test.shape}")

X_train shape: (1193, 100), X_test shape: (793, 100)


In [10]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

clf = LogisticRegression(max_iter=1000)
clf.fit(X_train, y_train)

y_pred = clf.predict(X_test)
acc = accuracy_score(y_test, y_pred)
print(f"Test accuracy using AvgWord2Vec features: {acc:.3f}\n")
print(classification_report(y_test, y_pred, target_names=news.target_names))

Test accuracy using AvgWord2Vec features: 0.950

                  precision    recall  f1-score   support

rec.sport.hockey       0.97      0.93      0.95       399
       sci.space       0.93      0.97      0.95       394

        accuracy                           0.95       793
       macro avg       0.95      0.95      0.95       793
    weighted avg       0.95      0.95      0.95       793



AvgWord2Vec features, despite being a very simple aggregation, give the logistic regression classifier enough signal to separate the two newsgroup topics well above chance — a dense, low-dimensional alternative to sparse BOW/TF-IDF features from topic 02.